# U09 交易與復原、現代資料庫速覽 ＋ ★報告規範

**資料庫管理**・11/12　<a href="https://colab.research.google.com/github/chang-ye-tu/db/blob/master/notebooks/unit09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

課程首頁：[github.com/chang-ye-tu/db](https://github.com/chang-ye-tu/db)・大綱：[syllabus.md](https://github.com/chang-ye-tu/db/blob/master/syllabus.md)・專題：[projects.md](https://github.com/chang-ye-tu/db/blob/master/projects.md)

把 U01 慘案 1、2 的債一次還清：**並行異常動物園**與**拔插頭實驗**——然後把你的專題送上台：★15 分鐘報告的完整規範。

> **投影片式 notebook 使用法**：上課跟著往下走，程式格按 `Shift+Enter` 執行；左側「目錄」可跳節。回家可以重跑、改參數做實驗——**講義是可以跑的**。
>
> 開始前建議：檔案 → 在雲端硬碟中儲存副本，改動才會留下來。

## 0. 本單元地圖（135 分鐘）

| 節 | 分鐘 | 內容 | 與專題的關係 |
|---|---|---|---|
| 第 1 節 | 50 | ACID 逐字・**並行異常動物園**（兩條連線）・隔離級別・鎖與死結・**MVCC／WAL 快照讀** | 你的競態 demo 背後的理論 |
| 第 2 節 | 50 | durability・**子行程當機模擬**（journal／WAL 兩種救法）・synchronous 取捨・**30 行玩具 WAL replay**・現代資料庫速覽 | 「當機不掉資料」的謎底 |
| ★報告規範 | 35 | 15 分鐘結構模板・demo 腳本化與三層備援・常見翻車・Q&A 與**評分官視角模擬** | **下次上課就上台** |

> 兩學期份量的引擎內部支線在此收官；自學延伸（LSM 玩具、Redis/Mongo/圖、Text-to-SQL）都在 `extra_modern.ipynb`。  
> 本單元新增的 SAVEPOINT、checkpoint 三模式與備份 API 均標為**選讀／加碼，不計入 135 分鐘主線**。

# 第 1 節：交易——ACID 逐字拆帳

| 字 | 承諾 | 這學期在哪見過 |
|---|---|---|
| **A**tomicity 原子性 | 整批全成或全不成 | U05 `with con:`、轉帳／購買函數 |
| **C**onsistency 一致性 | 交易前後規則都成立 | U02 約束＋U04 對帳查詢 |
| **I**solation 隔離性 | 並行的交易「像排隊一樣」 | U06 競態三武器——**今天拆原理** |
| **D**urability 持久性 | commit 了就死不了 | U01 慘案 2 的解藥——**今天拔插頭驗證** |

A 與 C 你已經很熟；今天主攻 **I**（第 1 節）與 **D**（第 2 節）。

In [ ]:
#@title 📦 本節道具：一張帳戶表（rollback-journal 模式先上場）
import sqlite3, os

for f in ("iso.db", "iso.db-journal", "iso.db-wal", "iso.db-shm"):
    if os.path.exists(f):
        os.remove(f)
con = sqlite3.connect("iso.db")
con.executescript("""
CREATE TABLE acct(id INTEGER PRIMARY KEY, owner TEXT, bal INTEGER CHECK (bal >= 0));
INSERT INTO acct VALUES (1, '佳蓉', 1000), (2, '威廷', 1000);
""")
con.commit()
print("iso.db 就緒：", con.execute("SELECT * FROM acct").fetchall())

In [ ]:
# 30 秒重溫 A 與 C：例外 → 自動回滾；約束 → 一致性的守門員（U05 學的，今天掛上理論）
def transfer(src, dst, amt):
    try:
        with con:
            con.execute("UPDATE acct SET bal = bal - ? WHERE id = ?", (amt, src))
            con.execute("UPDATE acct SET bal = bal + ? WHERE id = ?", (amt, dst))
        return "✅"
    except sqlite3.IntegrityError:
        return "❌（CHECK bal>=0 擋下，整筆回滾）"

print("轉 300：", transfer(1, 2, 300), "→", con.execute("SELECT id, bal FROM acct").fetchall())
print("轉 9999：", transfer(1, 2, 9999), "→", con.execute("SELECT id, bal FROM acct").fetchall())
con.execute("UPDATE acct SET bal = 1000"); con.commit()
print("（重置完畢。A=原子性、C=一致性 ✅——接下來兩節專攻 I 與 D）")

## 1.1 並行異常動物園

沒有隔離的世界會出哪些事？三隻經典怪物——前兩隻你其實都見過，今天給牠們掛上學名。

### 怪物一：lost update（更新遺失）——U01 慘案 1 的學名

In [ ]:
# 兩條連線做 read-modify-write：後寫的把先寫的蓋掉（記帳示範的正式版）
con_a = sqlite3.connect("iso.db"); con_b = sqlite3.connect("iso.db")
a = con_a.execute("SELECT bal FROM acct WHERE id = 1").fetchone()[0]     # 甲讀 1000
b = con_b.execute("SELECT bal FROM acct WHERE id = 1").fetchone()[0]     # 乙也讀 1000
con_a.execute("UPDATE acct SET bal = ? WHERE id = 1", (a - 120,)); con_a.commit()
con_b.execute("UPDATE acct SET bal = ? WHERE id = 1", (b - 80,));  con_b.commit()
final_bal = con.execute("SELECT bal FROM acct WHERE id = 1").fetchone()[0]
print(f"扣 120 又扣 80，餘額卻是 {final_bal}（正解 800）——甲的更新被「用舊值算的新值」蓋掉")
assert final_bal == 920
con_a.close(); con_b.close()
con.execute("UPDATE acct SET bal = 1000 WHERE id = 1"); con.commit()
print("→ 解藥你早就會：DB 端算術（bal = bal - ?）或條件式 UPDATE——把「讀改寫」壓成一句原子操作。")

### 怪物二、三：nonrepeatable read 與 phantom

- **nonrepeatable read**：同一交易裡讀同一列兩次，值變了（中間被別人改掉）。
- **phantom**：同一交易裡數同一批列兩次，**筆數**變了（中間被別人插入／刪除）。

決策型程式最怕這兩隻：「先查餘額夠不夠、再放行」的兩步之間，世界變了。
SQLite 怎麼治牠們？**快照**。切到 WAL 模式現場看：

In [ ]:
# MVCC／快照讀直播：讀者不擋寫者、讀者活在「開始交易那一刻」的世界
con.close()
wcon = sqlite3.connect("iso.db")
print("切換 journal_mode →", wcon.execute("PRAGMA journal_mode=WAL").fetchone()[0])

A = sqlite3.connect("iso.db"); B = sqlite3.connect("iso.db")
A.execute("BEGIN")
print("A（交易開始）讀到 bal =", A.execute("SELECT bal FROM acct WHERE id=1").fetchone()[0])
print("A 數帳戶數 =", A.execute("SELECT COUNT(*) FROM acct").fetchone()[0])

B.execute("UPDATE acct SET bal = 999999 WHERE id = 1")
B.execute("INSERT INTO acct VALUES (3, '亂入', 5)")
B.commit()
print("\n（B 已把 bal 改成 999999、還插了一個新帳戶，都 commit 了）\n")

print("A 再讀 bal =", A.execute("SELECT bal FROM acct WHERE id=1").fetchone()[0], "← 沒變！無 nonrepeatable read")
print("A 再數帳戶 =", A.execute("SELECT COUNT(*) FROM acct").fetchone()[0], "← 沒變！無 phantom")
A.commit()
print("A 結束交易後重讀 bal =", A.execute("SELECT bal FROM acct WHERE id=1").fetchone()[0], "← 新世界")
assert A.execute("SELECT COUNT(*) FROM acct").fetchone()[0] == 3
A.close(); B.close()
print()
print("→ 這就是 MVCC 精神（SQLite 以 WAL 快照實現）：每個讀交易拿到一張「開始瞬間的照片」，")
print("  寫者另起新版，互不打擾。報表跑一半資料不會在腳下變形——你的長報表最需要這個。")

In [ ]:
# 快照讀的免費紅利馬上用：把「多張報表」包進同一交易——中途任何寫入都影響不到你
rep = sqlite3.connect("iso.db")
rep.execute("BEGIN")                                   # 從這一刻起，rep 看到的世界被「拍照」
n_acct  = rep.execute("SELECT COUNT(*) FROM acct").fetchone()[0]
sum_bal = rep.execute("SELECT SUM(bal) FROM acct").fetchone()[0]
# （想像這中間還有五張報表、跑了三分鐘、別人一直在寫入……）
n_acct2 = rep.execute("SELECT COUNT(*) FROM acct").fetchone()[0]
rep.commit(); rep.close()
print(f"報表頭尾兩次數帳戶：{n_acct} == {n_acct2} ✅（同一張照片）")
print(f"總餘額 {sum_bal}——跨表加總也內部一致（不會出現「A 表已扣款、B 表還沒入帳」的鬼帳）")
assert n_acct == n_acct2
print("→ 專題的『統計報表』分頁：把整頁的查詢包進一個交易，月結數字才咬得死。")

In [ ]:
# 補一隻沒出場的怪物：dirty read（讀到別人「還沒 commit」的資料）——SQLite 直接不讓牠出生
wcon.execute("UPDATE acct SET bal = 1000 WHERE id = 1")      # 把上一格的實驗現場復原
wcon.execute("DELETE FROM acct WHERE id = 3")
wcon.commit()
A = sqlite3.connect("iso.db"); B = sqlite3.connect("iso.db")
B.execute("BEGIN"); B.execute("UPDATE acct SET bal = 77777 WHERE id = 1")     # B 改了但沒 commit
print("B 未提交的世界裡 bal =", B.execute("SELECT bal FROM acct WHERE id=1").fetchone()[0])
print("A 此刻讀到的 bal　　 =", A.execute("SELECT bal FROM acct WHERE id=1").fetchone()[0], "← 舊值！")
B.rollback()
print("B rollback 之後，77777 就像沒存在過——如果剛剛 A 讀得到它，A 就是根據幻覺做了決策。")
assert A.execute("SELECT bal FROM acct WHERE id=1").fetchone()[0] == 1000
A.close(); B.close()
print("→ dirty read 是最髒的異常，只有 READ UNCOMMITTED 這種級別才容忍；SQLite／主流預設都免疫。")
print("  也注意上面 B 自己讀得到 77777——「交易內看得見自己的未提交修改」是應該的（不算 dirty read）。")

In [ ]:
# 「先檢查再動作」的正確包法：把兩步關進同一個 BEGIN IMMEDIATE（U06 武器三的理論定位）
def withdraw_safe(id_, amt):
    c = sqlite3.connect("iso.db", timeout=1)
    try:
        c.execute("BEGIN IMMEDIATE")                    # 先佔寫權：檢查與動作之間沒人插得進來
        bal = c.execute("SELECT bal FROM acct WHERE id = ?", (id_,)).fetchone()[0]
        if bal < amt:
            c.rollback()
            return f"❌ 餘額只剩 {bal}"
        c.execute("UPDATE acct SET bal = bal - ? WHERE id = ?", (amt, id_))
        c.commit()
        return f"✅ 提了 {amt}"
    finally:
        c.close()

print(withdraw_safe(1, 600))
print(withdraw_safe(1, 600))                             # 第二次：餘額不夠 → 被檢查擋（而且擋得「準」）
wcon.execute("UPDATE acct SET bal = 1000 WHERE id = 1"); wcon.commit()
print("→ 檢查所讀到的值，在 commit 前保證不會被別人改——「兩步」在世界眼中變成「一步」。")

## 1.2 隔離級別總表（面試與教科書的共同考點）

| 級別 | lost update | nonrepeatable | phantom | 誰在用 |
|---|---|---|---|---|
| READ UNCOMMITTED | 😱 | 😱 | 😱 | 幾乎沒人（還會讀到未 commit 的髒資料） |
| READ COMMITTED | 部分防 | 😱 | 😱 | PostgreSQL／Oracle **預設** |
| REPEATABLE READ | ✅ | ✅ | 部分防 | MySQL InnoDB 預設 |
| SERIALIZABLE | ✅ | ✅ | ✅ | 「像完全排隊」——SQLite 給你的就是這級 |

- SQLite 簡單粗暴地達成 SERIALIZABLE：**一次只有一個寫者**（U06 的 BEGIN IMMEDIATE 排的就是這個隊）＋讀者看快照。
- 伺服器資料庫預設較弱（為了吞吐）——所以「應用層防競態」（U06 三武器）到了 PostgreSQL 一樣要寫；
  面試被問「你的系統怎麼防 lost update」，答案永遠是：**原子 UPDATE／唯一約束／鎖**，不是「資料庫會幫我」。

## 1.3 鎖、2PL 與死結（一頁看懂）

- 教科書解法：**兩階段鎖（2PL）**——交易先一路「拿鎖」、後一路「放鎖」，中間不得再拿；可證明可序列化。
- 拿鎖就可能**互等**＝死結。伺服器 DB 用「等待圖」偵測後挑一個犧牲者；SQLite 更乾脆——
  發現「你等我、我等你」的局，**立刻**讓一方收到 busy，連等都不等：

In [ ]:
# 死結現場（rollback-journal 模式的經典款：讀鎖想升級成寫鎖）
wcon.close()                                    # 離開 WAL 模式需要「所有連線都放手」——先收掉前面的連線
jcon = sqlite3.connect("iso.db")
jcon.execute("PRAGMA journal_mode=DELETE")      # 切回傳統 journal（WAL 下讀者不持鎖，演不出這齣）
jcon.close()

A = sqlite3.connect("iso.db", timeout=0.3); B = sqlite3.connect("iso.db", timeout=0.3)
A.execute("BEGIN"); A.execute("SELECT * FROM acct").fetchall()      # A：讀交易，持共享鎖
B.execute("BEGIN IMMEDIATE"); B.execute("UPDATE acct SET bal = bal + 1 WHERE id = 2")
print("A 持讀鎖、B 持寫鎖（都還沒 commit）——現在 A 也想寫：")
try:
    A.execute("UPDATE acct SET bal = bal + 1 WHERE id = 1")
except sqlite3.OperationalError as e:
    print(f"  A 立刻收到：{e}")
print("→ A 等 B 放行、B 的 commit 又等 A 放讀鎖——互等成局。SQLite 不玩等待遊戲，直接判 A 出局重來。")
A.rollback(); B.commit()
A.close(); B.close()
print("  App 端的正確反應：rollback → 稍等 → 重試（U06 的 with_retry() 就是為這寫的）。")

In [ ]:
# busy_timeout 這顆旋鈕單獨看：同一個撞鎖，0ms 立刻炸 vs 500ms 內自動等到
holder = sqlite3.connect("iso.db", check_same_thread=False)   # Timer 執行緒要替它 commit
holder.execute("BEGIN IMMEDIATE"); holder.execute("UPDATE acct SET bal = bal + 1 WHERE id = 1")

import threading, time
threading.Timer(0.2, holder.commit).start()            # 0.2 秒後放手

impatient = sqlite3.connect("iso.db")                  # 預設 timeout=5 但先歸零：完全不等
impatient.execute("PRAGMA busy_timeout = 0")
t0 = time.time()
try:
    impatient.execute("BEGIN IMMEDIATE")
except sqlite3.OperationalError as e:
    print(f"busy_timeout=0   ：{time.time()-t0:.2f}s 就炸 → {e}")
impatient.close()

patient = sqlite3.connect("iso.db")
patient.execute("PRAGMA busy_timeout = 500")           # 願意等 0.5 秒
t0 = time.time()
patient.execute("BEGIN IMMEDIATE")
print(f"busy_timeout=500 ：等了 {time.time()-t0:.2f}s 之後自動拿到 ✅")
patient.execute("ROLLBACK"); patient.close()
time.sleep(0.05); holder.close()
print("→ 大多數「database is locked」用一句 PRAGMA busy_timeout=3000 就消失——")
print("  它只是把「立刻放棄」改成「幫你等一下」；等不到才輪到 App 層重試（U06）。")

### §1 小結：三隻怪物 × 三件武器（把 U06 的直覺接上今天的理論）

| 怪物（學名） | 現場長相 | 武器 | 理論定位 |
|---|---|---|---|
| lost update | 讀改寫互相蓋 | DB 端算術／條件式 UPDATE | 把讀改寫壓成單一原子語句 |
| phantom／搶唯一位 | 檢查後冒出新列 | UNIQUE（含部分索引） | 約束在**任何**隔離級別都成立 |
| nonrepeatable（跨步決策） | 兩步之間世界變了 | BEGIN IMMEDIATE 包整段 | 手動把隔離範圍擴到整個流程 |

外加一層免費保險：SQLite 的快照讀讓**單一交易內**的報表天生一致。

### 隨堂練習：四個場景各是哪隻怪物的地盤？

① 選課系統兩人同秒搶最後一個名額　② 兩位管理員同時編輯同一筆商品
③ 月報表跑到一半，有人狂灌新訂單　④ 檢查「這時段沒被預約」之後才 INSERT

<details><summary>答案</summary>
① lost update 域（名額數量）→ 條件式 UPDATE。② lost update 的 UI 版 → 樂觀鎖 version（U06 第四武器）。
③ nonrepeatable/phantom → 免費解：把報表包進一個交易（快照讀）。④ 區間重疊不是 `UNIQUE` 能表達的規則 → 用 `BEGIN IMMEDIATE` 把「查重疊→INSERT」包成同一交易（U06 武器③）。
</details>

### 隨堂練習 A（3 分鐘）

1. 你的專題的核心操作，防的是三隻怪物中的哪一隻？用什麼武器？
2. 為什麼「先 SELECT 檢查、再 UPDATE」在 SERIALIZABLE 的 SQLite 也可能出事？（提示：你的兩步是**兩個交易**還是一個？）

<details><summary>答案方向</summary>

1. 扣庫存／名額＝lost update 域（原子 UPDATE）；搶唯一位＝phantom 域（UNIQUE 約束）；多步試算＝nonrepeatable 域（BEGIN IMMEDIATE 包整段）。
2. 隔離保護的是**單一交易內**的世界觀；而 Python `sqlite3` 的本課預設模式下，`SELECT` 本身不會隱式開啟交易。若沒明寫 `BEGIN`，檢查與動作之間就是無保護區——所以要嘛壓成一句、要嘛包進同一交易（回看 U05 §1.4）。
</details>

In [ ]:
# 練習 A 工作區：寫下你專題的「怪物分類表」（報告 Q&A 第 3 題的標準答案就從這來）
# 格式：操作名稱｜怪物學名｜武器｜最後防線（約束）
# 例：  借出圖書｜lost update｜條件式 UPDATE stock>=1｜CHECK(stock>=0)
# TODO：你的三個核心操作





## 1.4 選讀／加碼：巢狀 SAVEPOINT——只撤回交易的一小段

> 本小節不計入 135 分鐘主線。適合「一張主單含多個子操作，某個子操作失敗仍要保留其他成功部分」的流程。

```text
BEGIN                         外層交易
└─ SAVEPOINT order_scope      整張訂單
   ├─ SAVEPOINT item_1         子操作 1
   │  └─ RELEASE item_1       成功，把結果留在外層交易
   ├─ SAVEPOINT item_2         子操作 2
   │  └─ ROLLBACK TO item_2   失敗，只撤回 item_2 之後
   │     RELEASE item_2       `ROLLBACK TO` 不會刪掉 savepoint，所以再釋放
   └─ SAVEPOINT item_3         外層仍可繼續
      └─ RELEASE item_3
RELEASE order_scope
COMMIT                        訂單與成功的子操作一起落地
```

- `ROLLBACK` 會撤掉整個交易；`ROLLBACK TO name` 只倒帶到指定的 savepoint。
- `RELEASE` 是合併嵌套層級，**不等於久續化**；最外層 `COMMIT` 前當機，全部仍會撤回。

In [ ]:
# 選讀實驗：第 2 種商品庫存不足，只撤回這一行；訂單與其他兩行照常提交
for f in ("savepoint_lab.db", "savepoint_lab.db-journal", "savepoint_lab.db-wal", "savepoint_lab.db-shm"):
    if os.path.exists(f):
        os.remove(f)

sp = sqlite3.connect("savepoint_lab.db")
sp.execute("PRAGMA foreign_keys = ON")
sp.executescript("""
CREATE TABLE product(
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    stock INTEGER NOT NULL CHECK(stock >= 0)
);
CREATE TABLE orders(id INTEGER PRIMARY KEY, customer TEXT NOT NULL);
CREATE TABLE order_line(
    order_id INTEGER NOT NULL REFERENCES orders(id),
    product_id INTEGER NOT NULL REFERENCES product(id),
    qty INTEGER NOT NULL CHECK(qty > 0),
    PRIMARY KEY(order_id, product_id)
);
INSERT INTO product VALUES (1, '柚子茶', 5), (2, '烏龍茶', 1), (3, '咖啡豆', 2);
""")
sp.commit()

sp.execute("BEGIN")
sp.execute("SAVEPOINT order_scope")
sp.execute("INSERT INTO orders(customer) VALUES (?)", ("佳蓉",))
order_id = sp.execute("SELECT last_insert_rowid()").fetchone()[0]

for seq, (product_id, qty) in enumerate([(1, 2), (2, 3), (3, 1)], 1):
    mark = f"item_{seq}"                         # 名稱由程式內部的序號產生
    sp.execute(f"SAVEPOINT {mark}")
    try:
        sp.execute("INSERT INTO order_line VALUES (?, ?, ?)", (order_id, product_id, qty))
        cur = sp.execute("""UPDATE product SET stock = stock - ?
                            WHERE id = ? AND stock >= ?""", (qty, product_id, qty))
        if cur.rowcount != 1:
            raise ValueError(f"商品 {product_id} 庫存不足")
    except ValueError as e:
        sp.execute(f"ROLLBACK TO {mark}")         # 從 INSERT order_line 開始的這小段消失
        sp.execute(f"RELEASE {mark}")
        print("跳過：", e)
    else:
        sp.execute(f"RELEASE {mark}")
        print(f"商品 {product_id} 保留：{qty} 件")

sp.execute("RELEASE order_scope")
sp.commit()
lines = sp.execute("SELECT product_id, qty FROM order_line ORDER BY product_id").fetchall()
stocks = sp.execute("SELECT id, stock FROM product ORDER BY id").fetchall()
print("落地的明細：", lines)
print("最後庫存：", stocks)
assert lines == [(1, 2), (3, 1)]
assert stocks == [(1, 3), (2, 1), (3, 1)]
sp.close()
print("✅ 第 2 個子操作沒有汙染外層交易；這次的產品規格允許「跳過失敗項」。")

# 第 2 節：復原——「commit 了就死不了」是怎麼辦到的

## 2.1 問題的根源：延遲寫回 ＋ 部分寫入

U06 的 mini pager 賺了「同頁改百次只寫一次」，賭上「寫回前斷電就蒸發」。更慘的是**部分寫入**：
一個交易改了 5 頁，寫到第 3 頁時斷電——資料庫檔案「半新半舊」（U01 慘案 2 的資料庫版）。

解法只有一個思想：**先在別處留下完整的證據，再動本尊。**

| 流派 | 證據是什麼 | 當機後怎麼救 | SQLite |
|---|---|---|---|
| rollback journal | 改動前的**舊頁備份** | 把舊頁蓋回去＝**undo** 未完成的交易 | 預設模式 |
| WAL（write-ahead log） | 改動後的**新頁**先寫進 log | 把 log 裡**已 commit** 的重放進主檔＝**redo** | `PRAGMA journal_mode=WAL` |

口說無憑——**現場拔插頭**。用子行程 `os._exit()` 模擬「斷電」（連 Python 的收尾都來不及跑）：

In [ ]:
#@title ⚡ 拔插頭實驗一（rollback journal）：交易做到一半斷電 → undo 救回舊世界
import subprocess, sys

crash_script = '''
import sqlite3, os
con = sqlite3.connect("crash.db")
con.execute("PRAGMA journal_mode=DELETE")
con.execute("CREATE TABLE IF NOT EXISTS acct(id INTEGER PRIMARY KEY, bal INTEGER)")
con.execute("DELETE FROM acct")
con.execute("INSERT INTO acct VALUES (1, 1000), (2, 1000)")
con.commit()                                          # ← 這批已安全落地
con.execute("BEGIN")
con.execute("UPDATE acct SET bal = bal - 500 WHERE id = 1")          # 轉帳做到一半…
con.executemany("INSERT INTO acct(bal) VALUES (?)", [(i,) for i in range(60000)])
os._exit(1)                                           # ⚡ 斷電！（沒有 commit、沒有清理、什麼都沒有）
'''
for f in ("crash.db", "crash.db-journal"):
    if os.path.exists(f):
        os.remove(f)
subprocess.run([sys.executable, "-c", crash_script])
print("斷電後躺在磁碟上的檔案：", sorted(f for f in os.listdir(".") if f.startswith("crash")))
print("→ 那個 -journal 就是「熱 journal」：改動前的舊頁備份，還沒被清掉＝有交易死在半路")

c = sqlite3.connect("crash.db")                        # 重新開機：SQLite 開檔時發現熱 journal → 自動 undo
rows = c.execute("SELECT * FROM acct ORDER BY id").fetchall()
print("重開後的 acct：", rows)
assert rows == [(1, 1000), (2, 1000)]
c.close()
print("✅ 半路的 -500 與六萬筆插入全部蒸發、舊世界完好——「全有或全無」在斷電下也成立")

In [ ]:
#@title ⚡ 拔插頭實驗二（WAL）：commit 完立刻斷電（來不及歸檔）→ redo 救回新世界
crash_script2 = '''
import sqlite3, os
con = sqlite3.connect("wcrash.db")
con.execute("PRAGMA journal_mode=WAL")
con.execute("CREATE TABLE IF NOT EXISTS t(x INTEGER)")
con.execute("INSERT INTO t VALUES (42)")
con.commit()                                          # commit 成功＝新頁已寫進 -wal 並 fsync
os._exit(1)                                           # ⚡ 斷電！（-wal 還沒歸檔回主檔）
'''
for f in ("wcrash.db", "wcrash.db-wal", "wcrash.db-shm"):
    if os.path.exists(f):
        os.remove(f)
subprocess.run([sys.executable, "-c", crash_script2])
print("斷電後的檔案：", sorted(f for f in os.listdir(".") if f.startswith("wcrash")))
c = sqlite3.connect("wcrash.db")
print("重開後讀到：", c.execute("SELECT * FROM t").fetchall())
assert c.execute("SELECT * FROM t").fetchall() == [(42,)]
c.close()
print("✅ 42 還在——它當時只存在 -wal 裡，重開時被「重放（redo）」進主檔。")
print("   兩個實驗合起來：journal undo 未完成的、WAL redo 已承諾的——D（durability）的全部秘密。")

In [ ]:
# 順手觀察 WAL 檔的生活史：寫入讓 -wal 長大；TRUNCATE 成功才要求截為 0
wc = sqlite3.connect("iso.db")
wc.execute("PRAGMA journal_mode=WAL")
with wc:
    wc.executemany("INSERT INTO acct(owner, bal) VALUES (?, ?)",
                   [(f"臨時{i}", 1) for i in range(2000)])
size1 = os.path.getsize("iso.db-wal")
truncate_result = wc.execute("PRAGMA wal_checkpoint(TRUNCATE)").fetchone()
size2 = os.path.getsize("iso.db-wal") if os.path.exists("iso.db-wal") else 0
with wc:
    wc.execute("DELETE FROM acct WHERE owner LIKE '臨時%'")
remaining_temp = wc.execute("SELECT COUNT(*) FROM acct WHERE owner LIKE '臨時%'").fetchone()[0]
wc.execute("PRAGMA wal_checkpoint(TRUNCATE)"); wc.close()
print(f"寫入 2000 列後 -wal：{size1:,} bytes → TRUNCATE 後：{size2:,} bytes（只觀察）")
print("checkpoint 回傳：", truncate_result, "（頁數依環境而異，不寫死斷言）")
assert remaining_temp == 0
print("→ checkpoint 會把已提交 frame 併回主檔；只有成功的 TRUNCATE 模式才要求把檔案截為 0。")
print("  平時自動進行（每 ~1000 頁）；專題資料夾多出 -wal/-shm 檔是正常生活痕跡不是壞掉。")

## 2.2 選讀／加碼：WAL checkpoint 三模式

> 本小節不計入 135 分鐘主線。`checkpoint` 的工作是把 WAL 中可用的已提交 frame 寫回主檔；**寫回不代表 WAL 檔立刻變小**，已用空間可供後續重用。

| 模式 | 遇到讀者／寫者時 | 成功後的重點 | 適合場景 |
|---|---|---|---|
| `PASSIVE` | 不等待、不呼叫 busy handler；能做多少做多少 | 可能只 checkpoint 一部分，WAL 保持原大小 | 前景低干擾維運 |
| `FULL` | 透過 busy handler／timeout 等候寫者及舊快照讀者 | 成功時所有 frame 都寫回，但檔案仍可保留 | 確定主檔跟上 WAL |
| `TRUNCATE` | 如 `RESTART` 一樣完成 checkpoint 後，還要等現存讀者離開 WAL | 成功時將 `-wal` 截為 0 bytes | 打包、封存、想收回磁碟空間時 |

Python 讀到的回傳列是 `(busy, wal_frames, checkpointed_frames)`。`PASSIVE` 即使回傳 `busy=0` 也可能只做一部分，要一起看後兩欄；`FULL`／`TRUNCATE` 的 `busy=1` 表示被使用中的連線擋住。

In [ ]:
# 選讀實驗：先用舊快照讀者擋住 WAL，再比較 PASSIVE / FULL / TRUNCATE
for f in ("checkpoint_lab.db", "checkpoint_lab.db-wal", "checkpoint_lab.db-shm"):
    if os.path.exists(f):
        os.remove(f)

ck = sqlite3.connect("checkpoint_lab.db")
ck.execute("PRAGMA journal_mode = WAL")
ck.execute("PRAGMA wal_autocheckpoint = 0")            # 關掉自動 checkpoint，才看得到現場
ck.execute("PRAGMA busy_timeout = 150")                # 示範時不長等
ck.execute("CREATE TABLE event(id INTEGER PRIMARY KEY, note TEXT)")
ck.execute("INSERT INTO event(note) VALUES (?)", ("讀者的快照起點",))
ck.commit()

old_reader = sqlite3.connect("checkpoint_lab.db")
old_reader.execute("BEGIN")
snapshot_count = old_reader.execute("SELECT COUNT(*) FROM event").fetchone()[0]

with ck:
    ck.executemany("INSERT INTO event(note) VALUES (?)", [(f"後續事件 {i}",) for i in range(1, 201)])

def checkpoint(mode):
    row = ck.execute(f"PRAGMA wal_checkpoint({mode})").fetchone()
    print(f"{mode:8s} → busy={row[0]}, WAL frames={row[1]}, 已寫回={row[2]}")
    return row

print(f"舊讀者仍看到 {snapshot_count} 列；寫者已看到",
      ck.execute("SELECT COUNT(*) FROM event").fetchone()[0], "列")
blocked_rows = [checkpoint(mode) for mode in ("PASSIVE", "FULL", "TRUNCATE")]
print("→ 不斷言特定 frame 數：頁大小與版本會改變數字；觀察是否完成才是重點。")

old_reader.rollback()
old_reader.close()
print("\n舊快照已離開：")
full_after_release = checkpoint("FULL")
truncate_after_release = checkpoint("TRUNCATE")
final_count = ck.execute("SELECT COUNT(*) FROM event").fetchone()[0]
assert final_count == 201
assert all(len(row) == 3 for row in blocked_rows + [full_after_release, truncate_after_release])
assert full_after_release[0] == 0 and truncate_after_release[0] == 0
ck.close()
print("✅ 全程資料不變；checkpoint 調整的是「WAL 與主檔如何交接」，不是交易內容。")

## 2.3 代價的旋鈕：`synchronous`——每次 commit 要不要等 fsync

「commit 了就死不了」的成本是 **fsync**（強迫磁碟真的寫下去，不能只留在 OS 快取）。
`PRAGMA synchronous` 就是那顆旋鈕：

In [ ]:
import time

def per_commit_ms(pragmas, n=300):
    if os.path.exists("sync.db"):
        os.remove("sync.db")
    c = sqlite3.connect("sync.db")
    for p in pragmas:
        c.execute(p)
    c.execute("CREATE TABLE t(x)")
    t = time.time()
    for i in range(n):
        c.execute("INSERT INTO t VALUES (?)", (i,))
        c.commit()                                    # 一筆一 commit：durability 的單價
    c.close()
    return (time.time() - t) / n * 1000

for label, ps in [("DELETE ＋ FULL（最保險）", ["PRAGMA journal_mode=DELETE", "PRAGMA synchronous=FULL"]),
                  ("WAL ＋ NORMAL（建議日常）", ["PRAGMA journal_mode=WAL", "PRAGMA synchronous=NORMAL"]),
                  ("MEMORY ＋ OFF（裸奔）",     ["PRAGMA journal_mode=MEMORY", "PRAGMA synchronous=OFF"])]:
    print(f"{label:24s} {per_commit_ms(ps):7.3f} ms／commit")
print()
print("→ 本機是超快磁碟（甚至 RAM disk），差距溫和；傳統硬碟上 FULL 是「每秒幾十筆」等級——")
print("  差一百倍。裸奔模式當機會掉資料甚至壞檔，只配得上「可重算的暫存」。")
print("  Colab 跑專題：預設就好；批次匯入慢就想起 U05——把萬筆包進**一個**交易，共攤一次 fsync。")

### 隨堂練習：三個系統各轉哪一檔？

① 醫院掛號系統　② 感測器每秒一千筆的暫存緩衝（每小時彙整後即可拋棄）　③ 你的專題 demo

<details><summary>答案</summary>
① FULL（或 WAL+NORMAL）——掉一筆掛號是事故；② OFF/MEMORY 甚至 `:memory:`——資料可重來，吞吐優先；
③ 預設即可——評分不看這個，但你要**講得出**這顆旋鈕在換什麼（Q&A 加分題）。
安全與速度永遠在交易：重點是「知道自己選了什麼」。
</details>

## 2.4 教師現場實作：30 行玩具 WAL——把 redo 親手寫一遍

規格：一個 key-value 小庫。**任何修改先寫日誌**（append-only），commit 時補一筆 `COMMIT` 標記並落盤；
資料本體只活在記憶體。當機重開＝**重放日誌裡「有 COMMIT 標記」的交易**。

In [ ]:
import json

class ToyWAL:
    def __init__(self, path="toy.wal"):
        self.path = path
        self.data = {}                                  # 「資料庫本體」（記憶體）
        self.buffer = []                                # 目前交易的暫存
        self.recover()

    def set(self, k, v):
        self.buffer.append({"op": "set", "k": k, "v": v})     # 先記帳，不動本體

    def commit(self, txn_id):
        with open(self.path, "a", encoding="utf-8") as f:
            for op in self.buffer:
                f.write(json.dumps({"txn": txn_id, **op}, ensure_ascii=False) + "\n")
            f.write(json.dumps({"txn": txn_id, "op": "COMMIT"}) + "\n")
            f.flush(); os.fsync(f.fileno())             # ← durability 的那一下
        for op in self.buffer:                          # 落盤成功才動本體
            self.data[op["k"]] = op["v"]
        self.buffer = []

    def recover(self):
        if not os.path.exists(self.path):
            return
        txns, committed = {}, set()
        for line in open(self.path, encoding="utf-8"):
            try:
                e = json.loads(line)
            except json.JSONDecodeError:                # 斷電可能留下寫到一半的殘行
                break                                   # 殘行之後的都不可信——停止重放（真品也用 checksum 辨識有效尾端）
            if e["op"] == "COMMIT":
                committed.add(e["txn"])
            else:
                txns.setdefault(e["txn"], []).append(e)
        for t in sorted(committed):                     # 重放：只認有 COMMIT 標記的
            for op in txns.get(t, []):
                self.data[op["k"]] = op["v"]

if os.path.exists("toy.wal"):
    os.remove("toy.wal")
db = ToyWAL()
db.set("佳蓉", 1000); db.set("威廷", 1000); db.commit(txn_id=1)
print("交易 1 提交後：", db.data)

In [ ]:
# 當機劇本：交易 2 寫了一半日誌、還沒有 COMMIT 標記——直接對檔案動手模擬
with open("toy.wal", "a", encoding="utf-8") as f:
    f.write(json.dumps({"txn": 2, "op": "set", "k": "佳蓉", "v": 0}, ensure_ascii=False) + "\n")
    # ⚡ 斷電：COMMIT 標記永遠沒寫進去

db2 = ToyWAL()                                          # 重開機 → recover() 重放
print("重開後：", db2.data)
assert db2.data == {"佳蓉": 1000, "威廷": 1000}
print("✅ 交易 2 的半途改動被無視（沒有 COMMIT 標記＝沒發生過）——30 行就是 redo 的全部骨架。")

In [ ]:
# 更狠的當機劇本：連日誌本身都只寫了半行（斷電的真實長相）——recover 的容錯上場
with open("toy.wal", "a", encoding="utf-8") as f:
    f.write('{"txn": 3, "op": "set", "k": "威')          # ⚡ JSON 寫到一半就斷

db3 = ToyWAL()
print("半行殘骸也不怕，重開後：", db3.data)
assert db3.data == {"佳蓉": 1000, "威廷": 1000}
print("✅ recover() 讀到壞行就停止重放——「殘行之後皆不可信」。")
print("   真品 SQLite 在 WAL frame 存 rolling checksum 做完整性檢查：驗不過的尾端不算有效內容。")
print("   真品多的還有：checkpoint（歸檔）、頁級而非鍵級——原理你已經全懂了。")

### 隨堂練習：undo 還是 redo？

① 交易沒 commit 就當機　② commit 了、還沒歸檔回主檔就當機　③ commit 了、也歸檔了才當機

<details><summary>答案</summary>
① undo（journal 把舊頁蓋回去；WAL 模式其實更簡單——沒 COMMIT 標記的 frame 直接無視）。
② redo（WAL 重放已承諾的 frame——實驗二親眼看過）。③ 什麼都不用做：主檔已是最新。
一句總結：**日誌讓「commit 瞬間」成為世界的分界線**——之前的可以消失、之後的不許消失。
</details>

### 隨堂練習 B（3 分鐘）

1. `commit()` 裡「先寫日誌 fsync、再改 data」——兩步對調會發生什麼？
2. 日誌無限長怎麼辦？（提示：U06 看過的 `-wal` 檔沒有無限長——關鍵字 checkpoint）

<details><summary>答案</summary>

1. 對調＝先改本體再留證據：改到一半當機，本體半新半舊、日誌又不完整——兩頭落空。**Write-Ahead** 三個字就是順序本身。
2. checkpoint：把已提交的改動安全併回主檔。SQLite 預設約每 1000 頁觸發自動 checkpoint，但一般模式不保證 WAL 變小；讀者離開且 `TRUNCATE` 成功才會截成 0。
</details>

In [ ]:
# 練習 B-加碼工作區：幫 ToyWAL 補上 checkpoint()
# 規格：把 self.data 存成快照檔（json）、清空 wal；recover() 先載快照、再重放殘餘日誌
# TODO：
# def checkpoint(self): ...
# （寫完自測：set/commit → checkpoint → 再 set/commit 一筆 → 重開 → 兩批資料都在）


print("工作區就緒——寫完你就把 SQLite 的 wal_checkpoint 原理親手走過一遍了")

## 2.5 選讀／加碼：不要盲複製正在使用的 `.db`——兩種一致備份

> 本小節不計入 135 分鐘主線。WAL 模式下的最新 commit 可能還在 `-wal`；在程式運作中只複製主 `.db` 檔，得到的可能不是同一個時點的快照。

| 策略 | Python 入口 | 特性 | 適合時機 |
|---|---|---|---|
| SQLite Online Backup API | `source.backup(destination, pages=..., progress=...)` | 把一致快照複製到另一條連線；可分批複製與回報進度 | 應用仍在運作，要排程備份 |
| `VACUUM INTO` | `VACUUM INTO 'copy.db'` | 一個指令產生一致、壓實過的新檔；目標須不存在或為空檔 | 封存或交付前順便回收空頁 |

兩者都是**邏輯上的一致快照**，不會包含來源檔的後續新交易。備份完成之後還要能重開、對筆數，並跑 `PRAGMA integrity_check`——「有檔案」不等於「復原得了」。

In [ ]:
# 選讀實驗：同一個 WAL 來源，分別做 Backup API 與 VACUUM INTO，再實際重開驗證
backup_files = (
    "backup_lab_src.db", "backup_lab_src.db-wal", "backup_lab_src.db-shm",
    "backup_api.db", "backup_api.db-journal",
    "backup_vacuum.db", "backup_vacuum.db-journal",
)
for f in backup_files:
    if os.path.exists(f):
        os.remove(f)

src = sqlite3.connect("backup_lab_src.db")
src.execute("PRAGMA journal_mode = WAL")
src.executescript("""
CREATE TABLE product(id INTEGER PRIMARY KEY, name TEXT NOT NULL, price INTEGER NOT NULL);
CREATE INDEX idx_product_price ON product(price);
""")
src.executemany("INSERT INTO product VALUES (?, ?, ?)",
                [(i, f"商品 {i:03d}", 100 + (i % 17) * 10) for i in range(1, 301)])
src.execute("DELETE FROM product WHERE id % 4 = 0")     # 留一些可回收的空間
src.commit()

backup_progress = []
def remember_progress(status, remaining, total):
    backup_progress.append((remaining, total))

api_dst = sqlite3.connect("backup_api.db")
src.backup(api_dst, pages=8, progress=remember_progress)
api_dst.close()

src.execute("VACUUM INTO ?", ("backup_vacuum.db",))

def audit_backup(path):
    c = sqlite3.connect(path)
    metrics = c.execute("SELECT COUNT(*), SUM(price), MIN(id), MAX(id) FROM product").fetchone()
    integrity = c.execute("PRAGMA integrity_check").fetchone()[0]
    c.close()
    return metrics, integrity

source_audit = audit_backup("backup_lab_src.db")
api_audit = audit_backup("backup_api.db")
vacuum_audit = audit_backup("backup_vacuum.db")
print("來源／Backup API／VACUUM INTO 對帳：")
print(source_audit, api_audit, vacuum_audit, sep="\n")
print("檔案大小（只觀察，不當成跨平台斷言）：",
      {p: os.path.getsize(p) for p in ("backup_lab_src.db", "backup_api.db", "backup_vacuum.db")})
if backup_progress:
    print(f"Backup API progress callback：{len(backup_progress)} 次；最後 {backup_progress[-1]}")
assert source_audit == api_audit == vacuum_audit
assert source_audit[1] == "ok"
src.close()
print("✅ 兩份備份都能獨立重開、筆數與總價一致、integrity_check 通過。")

## 2.6 現代資料庫 30 秒速覽（詳細都在 `extra_modern.ipynb`）

| 家族 | 核心 idea | 一句話 |
|---|---|---|
| LSM-tree（RocksDB/Cassandra） | 寫入只 append，背景再合併排序 | 把隨機寫變循序寫——寫入猛獸 |
| 列式（DuckDB/ClickHouse） | 同欄放一起＋向量化 | U08 親測過：分析快 10–100 倍 |
| 文件（MongoDB） | 整包 JSON 當一筆 | schema 彈性；SQLite 也能玩（下一格） |
| 向量（FAISS/pgvector） | 語意→高維向量→最近鄰 | 「以文搜文」＝ AI 檢索（下下格） |

兩個 SQLite 就能嚐的：

In [ ]:
# 文件流：SQLite 的 JSON 函數——彈性欄位塞 JSON、查詢時再拆
jc = sqlite3.connect(":memory:")
jc.execute("CREATE TABLE events(id INTEGER PRIMARY KEY, payload TEXT)")   # payload 存 JSON 字串
jc.executemany("INSERT INTO events(payload) VALUES (?)", [
    ('{"type": "borrow", "user": {"name": "佳蓉"}, "book": "統計學習導論"}',),
    ('{"type": "return", "user": {"name": "威廷"}, "late_days": 3}',),
    ('{"type": "borrow", "user": {"name": "孟軒"}, "book": "資料庫概論"}',),
])
for r in jc.execute("""SELECT json_extract(payload, '$.type')       AS 類型,
                              json_extract(payload, '$.user.name')  AS 誰,
                              json_extract(payload, '$.late_days')  AS 逾期
                       FROM events"""):
    print(r)
print("\n→ 固定的欄位好好開欄（才能約束＋索引）；真正「每筆都不一樣」的雜項再進 JSON——混合式是實務常態。")

In [ ]:
# JSON 陣列也拆得動：json_each 把陣列展開成「一列一元素」——long format 又見面了
jc.execute("""INSERT INTO events(payload) VALUES
    ('{"type": "bulk_borrow", "user": {"name": "雅筑"}, "books": ["迴歸分析", "實驗設計", "SQL 聖經"]}')""")
for r in jc.execute("""SELECT json_extract(e.payload, '$.user.name') AS 誰, je.value AS 書
                       FROM events e, json_each(e.payload, '$.books') AS je"""):
    print(r)
print("\n→ json_each 是「表值函數」：一包陣列 → N 列。統計要的 long format、GROUP BY 都能接著做——")
print("  這正是 U04「一格塞清單」的救贖之路（如果你真的收到了這種資料）。")

In [ ]:
# 向量流：cosine 相似度的「以文搜文」玩具（真實世界把詞頻向量換成 embedding 模型）
import numpy as np

corpus = ["資料庫 課程 教 SQL 與 交易",
          "深夜 圖書館 的 推理 小說",
          "SQL 查詢 與 索引 效能 調校",
          "登山社 合歡山 行前 準備",
          "交易 與 復原 的 上課 筆記"]
vocab = sorted({w for s in corpus for w in s.split()})

def vec(s):
    v = np.array([s.split().count(w) for w in vocab], dtype=float)
    return v / (np.linalg.norm(v) or 1)

mat = np.array([vec(s) for s in corpus])
query = "想 複習 SQL 與 索引"
scores = mat @ vec(query)
ranked = np.argsort(-scores)
print(f"查詢：「{query}」")
for i in ranked[:3]:
    print(f"  {scores[i]:.3f}  {corpus[i]}")
assert ranked[0] == 2
print("\n→ 「相似」＝向量夾角小。把詞頻向量換成 LLM 的 embedding，就是 RAG 檢索的心臟——extra_modern 有完整版。")

# ★ 報告規範（35 分鐘）——下次上課就是你了

## R1. 15 分鐘的結構模板（12 講 ＋ 2 問 ＋ 1 換場，時間到即切）

| 分鐘 | 段落 | 要說什麼 | 常見錯 |
|---|---|---|---|
| 0–1.5 | 動機與情境 | 這系統給誰用、解決什麼麻煩 | 念題目規格（台下都讀過） |
| 1.5–4 | schema 與設計決策 | ER 圖一張＋**兩三個決策的「為什麼」** | 逐表逐欄唸 DDL |
| 4–10 | **live demo** | 走排練過的腳本（含競態亮點！） | 即興亂點、當場 debug |
| 10–12 | 報表解讀 | 挑 2 張最有故事的：「這張回答＿＿」 | 五張全講、只念數字 |
| 12–13 | 困難與 AI 協作 | 卡最久的一關＋AI 哪裡錯你怎麼抓 | 「一切順利沒什麼困難」 |
| 13–15 | Q&A | 深呼吸，§R4 演練過了 | 沒聽完問題就搶答 |

評分五面向（詳見 [projects.md](https://github.com/chang-ye-tu/db/blob/master/projects.md) §3）：
功能完成度・schema 與資料品質・報表分析深度・簡報與 demo・Q&A 程式理解。

## R2. demo 腳本化：把 6 分鐘寫成逐步清單

**demo 不是操作給大家看，是「說一個系統的故事」**。腳本格式（貼在專題 notebook 最上面）：

```
1. 開場畫面：報表分頁（先亮結果——全年借閱趨勢圖）
2. 流通分頁：借一本熱門書 → 指出庫存即時 -1
3. 【亮點】搶最後一本：跑競態 cell → 一人成功一人被好好拒絕
4. 觸發一個「應該失敗」：重複報名 → 指出人話錯誤訊息
5. 後台：新增一筆 → 下拉選單即時長出來
6. 回報表分頁：重新整理 → 剛剛的操作已入報表
```

三個鐵則：**每一步落點明確**（點哪裡、說哪句）；**亮點放中段**（觀眾最清醒）；**絕不現場改程式**。

## R3. 三層備援（共同要求第 10 條）

| 層 | 是什麼 | 何時啟用 |
|---|---|---|
| 一 | 報告前 30 分鐘在無痕視窗「全部執行」過的 Colab | 正常情況 |
| 二 | 報告前 30 分鐘新產生的 `share=True` 網址，開在第二個分頁（或第二台裝置） | 主分頁掛掉 |
| 三 | **完整走過一遍 demo 的截圖串／30 秒錄影** | 網路整個罷工 |

> 有備援照常計分；沒備援風險自負（共同要求第 10 條白紙黑字）。

## R4. Q&A 演練題庫（教師會從這類題抽）

1. 「`with con:` 到底做了什麼？拿掉會怎樣？」
2. 「你這個 UNIQUE 對應需求裡哪句話？」
3. 「兩個人同時按下這顆按鈕，最壞會發生什麼？你的防線在哪一層？」
4. 「這張報表的 window function 換成 GROUP BY 寫得出來嗎？為什麼（不）？」
5. 「你的索引為什麼建這幾個？最左前綴是什麼？」
6. 「合成資料的分佈假設是什麼？seed 換掉報表會差多少？」
7. 「（任指一段程式）這段在做什麼？這個參數拿掉會怎樣？」
8. 「AI 幫你寫的哪一段錯得最離譜？你怎麼發現的？」

**答不出來的處理**：誠實說「這段當時是這樣想的……我回去再驗證」比硬掰好——但**任指程式講不出來視同沒做**（AI 政策第 1 條）。



## R5. 評分官視角 mock scoring（10 分鐘，主線）

**這不是新的正式配分表**；只用 `2／1／0` 做模擬診斷：`2`＝報告當場有完整證據、`1`＝有做但證據不足、`0`＝缺席或現場失敗。不加總分，重點是找出**最可能拉低整體表現的單一個缺口**。

### 虛擬報告現場

- 系統可新增、查詢、取消預約；單人操作正常。但並行 demo 出現兩筆重複時段，schema 沒有對應的 `UNIQUE`。
- schema 有五張表、PK／FK／`NOT NULL`；日期字串同時出現 `2026-11-03` 與 `11/3`，也沒有驗證。
- 五張報表都會跑，含 join 與 window function；展示時只念出總數，沒有說「這張圖回答什麼」。
- demo 腳本實測 5 分 40 秒，主路徑順、備援截圖齊；「應該失敗」的路徑會顯示可讀訊息。
- Q&A 能解釋 `with con:` 與條件式 `UPDATE`，卻答不出來為何選現有索引，也沒有索引前後計時。

### 10 分鐘流程

1. **2 分**：獨立讀現場，把每個評分面向標記 `2／1／0`。
2. **4 分**：每個標記都要引用一條現場證據；不能只寫「感覺不錯」。
3. **3 分**：兩人比較。如果不同，問「報告現場哪個證據讓你改分？」
4. **1 分**：只選一個缺口，寫成今天可完成的修正動作。

<details><summary>校準方向（先做完再開）</summary>

- **功能完成度**：不宜滿標；並行下的核心規則已破。修正是 schema 約束＋失敗路徑，不是再補一個查詢頁。
- **schema 與資料品質**：有骨架但不宜滿標；日期格式不一致，且缺少可由 DB 保證的規則。
- **報表分析深度**：SQL 技巧存在，但「算出來」還沒變成「解釋出來」。
- **簡報與 demo**：現有證據最完整；時間、腳本、失敗路徑、備援都能當場觀察。
- **Q&A 程式理解**：有基礎但索引沒有實驗證據；補 `EXPLAIN QUERY PLAN` 與前後計時，並練一句「為何這個欄位順序」。

可接受的數字會因理由而異；**沒有現場證據的滿標才是這題要抓的錯**。
</details>

In [ ]:
# mock scoring 工作表：填 2 / 1 / 0、證據、今天能完成的修正（這不是正式配分）
mock_review = [
    {"aspect": "功能完成度",       "mark": "?", "evidence": "", "next_action": ""},
    {"aspect": "schema 與資料品質", "mark": "?", "evidence": "", "next_action": ""},
    {"aspect": "報表分析深度",     "mark": "?", "evidence": "", "next_action": ""},
    {"aspect": "簡報與 demo",        "mark": "?", "evidence": "", "next_action": ""},
    {"aspect": "Q&A 程式理解",      "mark": "?", "evidence": "", "next_action": ""},
]

allowed_marks = {"?", "0", "1", "2"}
assert all(item["mark"] in allowed_marks for item in mock_review)
print(f"{'aspect':<24} {'mark':<5} {'evidence':<24} next action")
print("-" * 82)
for item in mock_review:
    print(f"{item['aspect']:<22} {item['mark']:^5} {item['evidence']:<24} {item['next_action']}")
unfinished = sum(item["mark"] == "?" or not item["evidence"] or not item["next_action"]
                 for item in mock_review)
print(f"\n尚有 {unfinished} 個面向待完成。重點不是總分，而是「每個判斷都有證據＋有一個可執行修正」。")

## R6. 報告前檢查清單（現場以自己的 Colab 開啟報告；最終版 `.ipynb` 於報告結束後繳交）

- [ ] 無痕視窗開啟 → `全部執行` → 一鍵到底全綠（含 `launch()` 前的所有 cell）
- [ ] 固定 seed：重跑兩次，報表數字一模一樣
- [ ] 共同要求 10 條逐項自評表放在 notebook 開頭（✅/位置）
- [ ] 測試 cell 的 assert 全綠、「應該失敗」的有 ≥2 個
- [ ] AI 使用說明完整（工具／prompt 摘錄／錯誤與修正／驗證方式）
- [ ] demo 腳本貼在最上面；備援三層就位
- [ ] 報告用的 Colab 連結在無痕視窗自測開過一次（權限正確、可完整執行）

In [ ]:
# 小工具一：共同要求自評表產生器——跑一下、貼到你專題 notebook 的最上面
REQS = ["Schema：≥4 表、3NF、約束齊、附圖", "合成 ≥10,000 列、固定 seed、分佈擬真",
        "CRUD 全套、? 傳值、表單驗證", "交易保護＋併發競態示範",
        "報表 ≥5（≥1 window、≥2 join、≥1 圖）", "最慢查詢 EXPLAIN＋索引前後計時",
        "Gradio ≥3 分頁、接真資料庫", "≥8 assert（含 ≥2 應該失敗）",
        "AI 使用說明", "15 分鐘簡報＋demo 腳本＋三層備援"]
print("## 共同要求自評（貼進你的 notebook 開頭並逐項填寫）\n")
for i, req in enumerate(REQS, 1):
    print(f"- [ ] {i:2d}. {req}　→ 在第 ___ 節｜狀態：✅／🔜／❌")

In [ ]:
# 小工具二：報告排練計時器——把你的段落秒數填進來，看會不會爆時
segments = [("動機與情境", 1.5), ("schema 與決策", 2.0), ("live demo", 5.5),
            ("報表解讀", 1.5), ("困難與 AI", 1.0)]      # 合計 11.5——留 0.5 分鐘的呼吸空間
total = 0
print(f"{'段落':<12s}{'長度':>6s}{'累計':>8s}")
for label, mins in segments:
    total += mins
    print(f"{label:<12s}{mins:>5.1f}m{total:>7.1f}m")
print(f"\n講述合計 {total:.1f} 分鐘（上限 12；留 2 分 Q&A、1 分換場）",
      "✅ 安全" if total <= 12 else "🔥 會被切——刪內容，不要講快一點（講快＝觀眾聽不懂）")
assert total <= 12
# 排練時拿手機逐段核對：哪段實測超過表定 1.3 倍，就是該剪的段

## R7. demo 翻車排行榜（歷屆與他校觀察，前車之鑑）

| # | 翻車 | 預防 |
|---|---|---|
| 1 | Colab runtime 中午被回收，上台才發現全空 | 報告前 30 分鐘重跑（R3 第一層） |
| 2 | 現場手滑改到程式 → 全場看你 debug | demo **絕不碰程式格**，只碰 UI |
| 3 | `share=True` 網址過期／投影機沒網路 | 第三層備援（截圖／錄影）永遠開著 |
| 4 | 示範「應該失敗」時真的炸出 traceback | 錯誤處理沒接好——交件前把每條失敗路徑點一遍 |
| 5 | 報表圖中文變 □□ | 字型 bootstrap（U01 刀四）放 notebook 開頭、報告前跑過 |
| 6 | 講太快提前 3 分鐘講完，Q&A 變 5 分鐘 | 排練計時（小工具二）；寧可從容講 11.5 分 |

## 課堂實作（35 分內含在報告規範中）：兩人互演

1.（10 分）完成 R5 的評分官視角 mock scoring——每個判斷都引用現場證據；
2.（10 分）互當觀眾走一次 demo 腳本——計時、記下「哪一步你看不懂我在幹嘛」；
3.（10 分）互抽 R4 的三題 Q&A——答不順的寫進待辦；
4.（5 分）交換跑對方的 notebook「全部執行」——別人的電腦是最誠實的測試環境。

## 專題進度建議（非繳交）——最後一哩

- 今天回家：完成 R6 清單前四項；
- 報告前三天：錄好第三層備援、把 demo 腳本走三遍（一遍計時、一遍給室友看、一遍無痕視窗）；
- 報告前一天：早點睡。系統做完了，剩下的是把它「說好」。

### FAQ：專題到底要不要開 WAL？

- **可開可不開**：單人操作的 Colab 專題，預設模式完全夠用；評分不看這個。
- **想開的理由**：demo 併發亮點時「讀不擋寫」更順；`PRAGMA journal_mode=WAL` 一句話的事。
- **開了之後**：資料夾多出 `-wal`、`-shm` 是正常。要封存時優先用 Backup API／`VACUUM INTO`；若要取得單一 `.db`，先結束其他讀寫連線，再確認 `PRAGMA wal_checkpoint(TRUNCATE)` 成功。
- **真正要記的**：正常 durability 設定下，commit 是恢復時的分界；所選 `synchronous` 會決定這個承諾有多強。

# 本單元你應該帶走

1. ACID 的 I 與 D 補完：**快照讀（MVCC）**讓讀者活在開始交易那一刻——nonrepeatable 與 phantom 消失；lost update 仍要靠你的原子寫法。
2. 隔離級別是「吞吐 vs 異常」的滑桿；SQLite 給 SERIALIZABLE（單寫者＋快照），伺服器 DB 預設更弱——**應用層防線永遠要寫**。
3. 死結＝互等成局；SQLite 立刻判一方出局，App 收到 busy 就 rollback＋重試。
4. （選讀）`SAVEPOINT` 讓嵌套子操作可局部撤回；`ROLLBACK TO` 後仍要 `RELEASE`，最外層 commit 才算落地。
5. durability＝**先留證據再動本尊**：journal 存舊頁（undo 未完成）、WAL 存新頁（redo 已承諾）——兩個拔插頭實驗親眼看過。
6. （選讀）checkpoint 不等於縮檔：`PASSIVE`／`FULL` 可寫回主檔仍保留 WAL；`TRUNCATE` 成功才將它截為 0。
7. fsync 是 durability 的單價：`synchronous` 旋鈕、批次共攤（U05 的萬筆一交易）。
8. 30 行玩具 WAL：沒有 COMMIT 標記＝沒發生過；壞行之後皆不可信（checksum 的精神）——Write-Ahead 的順序就是全部。
9. （選讀）Backup API 與 `VACUUM INTO` 會產生一致快照；備份後要重開、對筆數、做 integrity check。
10. 報告：結構模板、腳本化 demo、三層備援、Q&A 與評分官視角演練——每個判斷都要有現場證據。

**接下來**：專題報告 I–VI。自學延伸：`extra_modern.ipynb`（LSM 玩具、Redis/Mongo/圖、SQLite JSON 進階、向量檢索、Text-to-SQL）。讀物：Silberschatz／Korth／Sudarshan ch17–19 選讀；Garcia-Molina／Ullman／Widom ch17–18 選讀。

---
## 附錄 A：本單元 cheatsheet

```sql
-- 交易與隔離
BEGIN / BEGIN IMMEDIATE / COMMIT / ROLLBACK
SAVEPOINT item_1;
ROLLBACK TO item_1;                    -- 只倒帶到這個層級，savepoint 仍存在
RELEASE item_1;                        -- 合併／移除這個層級
PRAGMA busy_timeout = 3000;            -- 撞鎖等一下（配 App 端重試）
PRAGMA journal_mode = WAL;             -- 讀不擋寫＋快照讀（單機 App 建議）
PRAGMA synchronous = NORMAL;           -- WAL 下的甜蜜點；FULL 最保險；OFF 只配暫存
PRAGMA wal_checkpoint(PASSIVE);        -- 不等；能寫回多少做多少
PRAGMA wal_checkpoint(FULL);           -- 等待後完整寫回，不保證縮檔
PRAGMA wal_checkpoint(TRUNCATE);       -- 讀寫者允許且成功時截為 0
VACUUM INTO 'snapshot.db';             -- 新建一份壓實的一致快照

-- JSON（文件流）
json_extract(payload, '$.a.b')         -- 取值
FROM t, json_each(t.payload, '$.arr')  -- 陣列展開成一列一元素
```

```python
# 一致備份（destination 是另一條 sqlite3.Connection）
source.backup(destination, pages=100, progress=callback)

# 防三怪物的武器對照（U06 學的，今天補上學名）
lost update        → UPDATE t SET x = x - ?（DB 端算術）／條件式 UPDATE／樂觀鎖 version
phantom／搶唯一位  → UNIQUE 約束（含部分索引）
多步讀寫不一致     → BEGIN IMMEDIATE 包整段 ＋ busy 重試
報表要一致         → 包進一個交易（快照讀免費送）
```

## 附錄 B：讀物地圖（本單元）

| 講義小節 | Silberschatz 7e | Garcia-Molina/Ullman/Widom 2e |
|---|---|---|
| ACID、隔離級別 | §17.1–17.5 | §18.1 |
| 鎖、2PL、死結 | §18.1–18.3 | §18.3–18.4 |
| MVCC／快照 | §18.7–18.8 | §18.8 |
| 日誌與復原（undo/redo） | §19.1–19.4 | §17.2–17.4 |
| WAL | https://sqlite.org/wal.html | —— |